<a href="https://colab.research.google.com/github/sayja-yug/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sayja-yug/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

### Lane 2 — Refresh / Content Opportunity Scoring

My lane is **Refresh / Content Opportunity Scoring**.

The decision I want to support is:

> Which content pages should a reviewer inspect first for refresh, expansion, protection, pruning, or monitoring?

The Week-4 baseline already provides a transparent starting point using observable search signals such as `gsc_impressions`, `gsc_clicks`, and `gsc_avg_position`. It produces a `baseline_score`, `reason_code`, and `action_label`.

For Week 5, I will test whether a learned model can identify useful patterns that a fixed rule may miss.

### Why I am using a supervised model

I will use a supervised classification/ranking approach only if I can define an observed outcome that is separate from the Week-4 rule.

The model must learn from observable inputs rather than learning the Week-4 `baseline_score`, `reason_code`, or `action_label`.

Those Week-4 outputs will therefore be treated as **baseline outputs for comparison**, not as model features.

### Candidate model

My primary model will be **Random Forest**, with a simpler model such as Logistic Regression used as a reference where practical.

I chose Random Forest because Lane 2 can contain nonlinear relationships between search visibility, clicks, position, sessions, and engagement. A tree ensemble can represent interactions and threshold effects without requiring me to specify every interaction manually.

I will not assume that Random Forest is better simply because it is more complex. I will compare its validation performance against the Week-4 baseline using the same evaluation definition.

### Candidate observable features

The model will use only signals that are available at the decision point and are not derived from the Week-4 decision itself.

Examples include:

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_pageviews`
- `ga4_sessions`
- `ga4_users`
- `ga4_engaged_sessions`
- `ga4_total_engagement_sec`
- `sessions_organic`
- `sessions_direct`
- `sessions_referral`
- `sessions_social`
- `sessions_paid`
- `sessions_ai`
- `scroll_events`

Identifiers such as `client_hash_id` and `content_hash_id` will be used for grouping, joining, or tracing rows when necessary, but they will not be used as predictive features.

### What I will not use as features

I will explicitly exclude:

- `baseline_score`
- `reason_code`
- `action_label`

because these are outputs of my Week-4 baseline.

I will also exclude any future-window measurements or fields that contain information from after the decision point.

### Success criterion

The model earns its place only if it provides useful improvement over the Week-4 baseline on the same evaluation setup.

Because Lane 2 produces a ranked review queue, I will focus on ranking-oriented evaluation such as **Precision@K** and, where appropriate, average precision.

The goal is not to maximize model complexity. The goal is to determine whether a learned model produces a more useful review ranking than the transparent Week-4 rule.

### Claim boundary

This model will provide **decision support**, not proof that refreshing a page will cause recovery.

A high-ranked page is a candidate for human review, not a guaranteed successful refresh.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# WEEK 5 — SECTION 1
# Lane 2: Refresh / Content Opportunity Scoring
# Method setup and feature audit
# ============================================================

import os
import pandas as pd
import numpy as np

print("=" * 70)
print("WEEK 5 — SECTION 1: METHOD CHOICE AND FEATURE AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Locate Week-4 baseline CSV
# ------------------------------------------------------------

possible_paths = [
    "/content/work/outputs/baseline_action_score.csv",
    "work/outputs/baseline_action_score.csv",
    "/content/baseline_action_score.csv",
    "baseline_action_score.csv"
]

baseline_path = None

for path in possible_paths:
    if os.path.exists(path):
        baseline_path = path
        break

if baseline_path is None:
    raise FileNotFoundError(
        "Week-4 baseline_action_score.csv was not found.\n"
        "Place the file at work/outputs/baseline_action_score.csv "
        "or upload it to Colab."
    )

print(f"\n✅ Week-4 baseline found:")
print(baseline_path)


# ------------------------------------------------------------
# 2. Load Week-4 baseline
# ------------------------------------------------------------

baseline_df = pd.read_csv(baseline_path)

print("\nBaseline shape:")
print(baseline_df.shape)

print("\nBaseline columns:")
print(baseline_df.columns.tolist())


# ------------------------------------------------------------
# 3. Check the required Week-4 baseline outputs
# ------------------------------------------------------------

required_baseline_columns = [
    "baseline_score",
    "reason_code",
    "action_label"
]

print("\n" + "=" * 70)
print("CHECKING WEEK-4 BASELINE OUTPUT COLUMNS")
print("=" * 70)

for col in required_baseline_columns:
    if col in baseline_df.columns:
        print(f"✅ {col}")
    else:
        print(f"❌ {col} MISSING")


# ------------------------------------------------------------
# 4. Define candidate observable features
# ------------------------------------------------------------

candidate_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events"
]

print("\n" + "=" * 70)
print("CANDIDATE OBSERVABLE FEATURES")
print("=" * 70)

available_features = []
missing_features = []

for col in candidate_features:
    if col in baseline_df.columns:
        available_features.append(col)
        print(f"✅ {col}")
    else:
        missing_features.append(col)
        print(f"⚠️ {col} not available in Week-4 CSV")


# ------------------------------------------------------------
# 5. Explicitly exclude baseline outputs
# ------------------------------------------------------------

excluded_from_model = [
    "baseline_score",
    "reason_code",
    "action_label",
    "client_hash_id",
    "content_hash_id",
    "report_date"
]

print("\n" + "=" * 70)
print("EXCLUDED FROM MODEL FEATURES")
print("=" * 70)

for col in excluded_from_model:
    if col in baseline_df.columns:
        print(f"🚫 {col}")


# ------------------------------------------------------------
# 6. Check that no baseline output accidentally appears
#    inside our candidate feature list
# ------------------------------------------------------------

leakage_check = set(available_features).intersection(
    set(["baseline_score", "reason_code", "action_label"])
)

print("\n" + "=" * 70)
print("BASELINE-OUTPUT LEAKAGE CHECK")
print("=" * 70)

if len(leakage_check) == 0:
    print("✅ No Week-4 baseline output is being used as a model feature.")
else:
    print("❌ POTENTIAL LEAKAGE:")
    print(leakage_check)


# ------------------------------------------------------------
# 7. Show baseline ranking
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("WEEK-4 BASELINE TOP 10")
print("=" * 70)

display(
    baseline_df[
        [
            col for col in [
                "report_date",
                "client_hash_id",
                "content_hash_id",
                "baseline_score",
                "reason_code",
                "action_label"
            ]
            if col in baseline_df.columns
        ]
    ].head(10)
)


# ------------------------------------------------------------
# 8. Baseline score summary
# ------------------------------------------------------------

if "baseline_score" in baseline_df.columns:

    print("\n" + "=" * 70)
    print("BASELINE SCORE SUMMARY")
    print("=" * 70)

    print(baseline_df["baseline_score"].describe())


# ------------------------------------------------------------
# 9. Action distribution
# ------------------------------------------------------------

if "action_label" in baseline_df.columns:

    print("\n" + "=" * 70)
    print("WEEK-4 ACTION LABEL DISTRIBUTION")
    print("=" * 70)

    display(
        baseline_df["action_label"]
        .value_counts(dropna=False)
        .to_frame("count")
    )


# ------------------------------------------------------------
# 10. Final Section-1 status
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SECTION 1 STATUS")
print("=" * 70)

print(f"Rows available: {len(baseline_df):,}")
print(f"Candidate features available: {len(available_features)}")
print(f"Candidate features missing: {len(missing_features)}")

if len(leakage_check) == 0:
    print("✅ Section 1 feature audit passed.")
else:
    print("❌ Section 1 requires leakage cleanup before continuing.")

print("\nNext step:")
print("SECTION 2 — Split design and observed target definition")


WEEK 5 — SECTION 1: METHOD CHOICE AND FEATURE AUDIT

✅ Week-4 baseline found:
/content/baseline_action_score.csv

Baseline shape:
(7928488, 11)

Baseline columns:
['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'impression_score', 'position_score', 'baseline_score', 'reason_code', 'action_label']

CHECKING WEEK-4 BASELINE OUTPUT COLUMNS
✅ baseline_score
✅ reason_code
✅ action_label

CANDIDATE OBSERVABLE FEATURES
✅ gsc_impressions
✅ gsc_clicks
✅ gsc_avg_position
⚠️ ga4_pageviews not available in Week-4 CSV
⚠️ ga4_sessions not available in Week-4 CSV
⚠️ ga4_users not available in Week-4 CSV
⚠️ ga4_engaged_sessions not available in Week-4 CSV
⚠️ ga4_total_engagement_sec not available in Week-4 CSV
⚠️ sessions_organic not available in Week-4 CSV
⚠️ sessions_direct not available in Week-4 CSV
⚠️ sessions_referral not available in Week-4 CSV
⚠️ sessions_social not available in Week-4 CSV
⚠️ sessions_paid not available in Week-4 CSV
⚠️

,report_date,client_hash_id,content_hash_id,baseline_score,reason_code,action_label
0,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,0.600067,HIGH_IMPRESSIONS,MONITOR
1,2026-03-29,client_e547b89c05043229,content_eadb33b5df496f4a,0.590105,HIGH_IMPRESSIONS,MONITOR
2,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,0.586040,HIGH_IMPRESSIONS,MONITOR
3,2026-03-28,client_e547b89c05043229,content_eadb33b5df496f4a,0.577096,HIGH_IMPRESSIONS,MONITOR
4,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,0.566264,HIGH_IMPRESSIONS,MONITOR
5,2026-03-30,client_e547b89c05043229,content_eadb33b5df496f4a,0.531705,HIGH_IMPRESSIONS,MONITOR
6,2026-03-27,client_e547b89c05043229,content_eadb33b5df496f4a,0.522913,HIGH_IMPRESSIONS,MONITOR
7,2026-03-31,client_e547b89c05043229,content_eadb33b5df496f4a,0.519803,HIGH_IMPRESSIONS,MONITOR
8,2026-03-24,client_e547b89c05043229,content_eadb33b5df496f4a,0.504364,HIGH_IMPRESSIONS,MONITOR
9,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,0.499841,HIGH_IMPRESSIONS,MONITOR



BASELINE SCORE SUMMARY
count    7.928488e+06
mean     1.013774e-02
std      1.129560e-02
min      6.024096e-03
25%      6.024096e-03
50%      6.024096e-03
75%      6.500546e-03
max      6.000669e-01
Name: baseline_score, dtype: float64

WEEK-4 ACTION LABEL DISTRIBUTION


,count
action_label,
MONITOR,7928487
NaN,1



SECTION 1 STATUS
Rows available: 7,928,488
Candidate features available: 3
Candidate features missing: 12
✅ Section 1 feature audit passed.

Next step:
SECTION 2 — Split design and observed target definition


## 2. Split design and observed target

### Decision point

For Lane 2, I want to prioritize content pages that deserve human review for refresh or improvement.

I will use March 2026 as the feature/decision window and April 2026 as the future outcome window.

The model therefore follows:

**March 2026 observable signals → April 2026 observed outcome**

This keeps the future target separate from the information available when the decision would have been made.

### Feature window

The model features will be calculated from March 2026 data.

Candidate signals include:

- GSC impressions
- GSC clicks
- GSC average position
- GA4 sessions and engagement metrics where available
- other observable performance signals available before the decision point

The Week-4 `baseline_score`, `reason_code`, and `action_label` will NOT be used as model features because they are outputs of the baseline rule.

### Target definition

The target is an observed future decline indicator.

A page is labelled `future_decline = 1` when:

1. it has enough March search volume to make the comparison meaningful;
2. it has at least a minimum number of March clicks;
3. April clicks are at least 20% lower than March clicks.

Otherwise the page is labelled `future_decline = 0`.

This is a proxy for identifying pages that subsequently experienced a meaningful performance decline. It is not a claim that refreshing the page would necessarily recover the lost traffic.

### Validation design

Because the target is measured after the feature window, the split must respect time.

I will not randomly mix March and April information.

The model will learn only from the March feature window and will be evaluated against the April observed outcome.

The Week-4 baseline will be evaluated against the SAME April outcome and on the SAME set of eligible pages.

This makes the model-versus-baseline comparison fair.

### Leakage controls

I will exclude:

- April metrics from the March feature matrix;
- `baseline_score`;
- `reason_code`;
- `action_label`;
- client/content identifiers as predictive features.

The client and content identifiers may be retained only for joining, grouping, and tracing results.

### Why this split is appropriate

A random split would allow observations from the same time period to appear on both sides of validation and would not represent the real decision process.

A time-based split better matches the intended use:

> observe the page now → make a prioritization decision → observe what happens later.

The model will therefore be judged on whether its March-based ranking identifies pages that subsequently show the defined April decline.

In [2]:
# ============================================================
# WEEK 5 — COLAB ENVIRONMENT REPAIR
# Run this cell ONCE.
# ============================================================

!pip uninstall -y pandas huggingface_hub datasets -q

!pip install --no-cache-dir --force-reinstall \
    pandas==2.2.2 \
    huggingface_hub==0.33.5 \
    datasets==3.6.0 \
    pyarrow==18.1.0 \
    -q

print("============================================================")
print("PACKAGE REPAIR COMPLETE")
print("============================================================")
print("Pandas       : 2.2.2")
print("HuggingFace  : 0.33.5")
print("Datasets     : 3.6.0")
print("PyArrow      : 18.1.0")
print()
print("IMPORTANT:")
print("Restart the Colab runtime now.")
print("Do NOT run the next cells before restarting.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 201.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.0/104.0 kB 59.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 215.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.7/515.7 kB 271.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 289.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 MB 235.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 267.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 335.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 238.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 306.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 279.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# ============================================================
# WEEK 5 — LOAD WAREHOUSE DATA FOR SECTION 2
#
# Lane 2: Refresh / Content Opportunity Scoring
#
# March 2026 = feature / decision window
# April 2026 = future observed outcome window
# ============================================================

import pandas as pd
from google.colab import userdata
from huggingface_hub import HfFileSystem

print("=" * 75)
print("WEEK 5 — SECTION 2 DATA LOAD")
print("LANE 2: REFRESH / CONTENT OPPORTUNITY SCORING")
print("=" * 75)


# ------------------------------------------------------------
# 1. Load Hugging Face token
# ------------------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError(
        "HF_TOKEN was not found in Colab Secrets. "
        "Add your Hugging Face READ token as HF_TOKEN."
    )

print("✅ Hugging Face token loaded successfully.")


# ------------------------------------------------------------
# 2. Connect to Hugging Face
# ------------------------------------------------------------

fs = HfFileSystem(token=HF_TOKEN)

DATASET_REPO = "datasets/FlyRank/internship-warehouse"

FACT_PATH = (
    DATASET_REPO
    + "/fact_content_daily_performance"
)

print("✅ Connected to Hugging Face.")
print("Dataset:", DATASET_REPO)
print("Fact table:", FACT_PATH)


# ------------------------------------------------------------
# 3. Find March and April parquet partitions
# ------------------------------------------------------------

march_pattern = (
    FACT_PATH + "/month=2026-03/*.parquet"
)

april_pattern = (
    FACT_PATH + "/month=2026-04/*.parquet"
)

march_files = fs.glob(march_pattern)
april_files = fs.glob(april_pattern)


print("\n" + "=" * 75)
print("PARTITION CHECK")
print("=" * 75)

print("March parquet files:", len(march_files))
print("April parquet files:", len(april_files))


if len(march_files) == 0:
    raise FileNotFoundError(
        "March 2026 partition was not found."
    )

if len(april_files) == 0:
    raise FileNotFoundError(
        "April 2026 partition was not found."
    )

print("✅ March 2026 partition found.")
print("✅ April 2026 partition found.")


# ------------------------------------------------------------
# 4. Show example files
# ------------------------------------------------------------

print("\nExample March files:")

for f in march_files[:3]:
    print(" ", f)

print("\nExample April files:")

for f in april_files[:3]:
    print(" ", f)


# ------------------------------------------------------------
# 5. Convert paths to Hugging Face URLs
# ------------------------------------------------------------

march_urls = [
    "hf://" + f
    for f in march_files
]

april_urls = [
    "hf://" + f
    for f in april_files
]


# ------------------------------------------------------------
# 6. Only load columns required for Section 2
# ------------------------------------------------------------

REQUIRED_COLUMNS = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

print("\n" + "=" * 75)
print("COLUMNS USED FOR SECTION 2")
print("=" * 75)

for col in REQUIRED_COLUMNS:
    print("•", col)


# ------------------------------------------------------------
# 7. Load March 2026
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("READING MARCH 2026")
print("=" * 75)

march_parts = []

for i, url in enumerate(march_urls, start=1):

    print(
        f"Loading March file {i}/{len(march_urls)}..."
    )

    part = pd.read_parquet(
        url,
        columns=REQUIRED_COLUMNS,
        storage_options={
            "token": HF_TOKEN
        }
    )

    march_parts.append(part)


march_daily = pd.concat(
    march_parts,
    ignore_index=True
)

print("\n✅ March data loaded.")
print("March shape:", march_daily.shape)


# ------------------------------------------------------------
# 8. Load April 2026
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("READING APRIL 2026")
print("=" * 75)

april_parts = []

for i, url in enumerate(april_urls, start=1):

    print(
        f"Loading April file {i}/{len(april_urls)}..."
    )

    part = pd.read_parquet(
        url,
        columns=REQUIRED_COLUMNS,
        storage_options={
            "token": HF_TOKEN
        }
    )

    april_parts.append(part)


april_daily = pd.concat(
    april_parts,
    ignore_index=True
)

print("\n✅ April data loaded.")
print("April shape:", april_daily.shape)


# ------------------------------------------------------------
# 9. Standardize dates
# ------------------------------------------------------------

march_daily["report_date"] = pd.to_datetime(
    march_daily["report_date"],
    errors="coerce"
)

april_daily["report_date"] = pd.to_datetime(
    april_daily["report_date"],
    errors="coerce"
)


# ------------------------------------------------------------
# 10. Verify required columns
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("REQUIRED COLUMN CHECK")
print("=" * 75)

missing_march = [
    c for c in REQUIRED_COLUMNS
    if c not in march_daily.columns
]

missing_april = [
    c for c in REQUIRED_COLUMNS
    if c not in april_daily.columns
]


if missing_march:
    raise ValueError(
        "Missing March columns: "
        + str(missing_march)
    )

if missing_april:
    raise ValueError(
        "Missing April columns: "
        + str(missing_april)
    )


for col in REQUIRED_COLUMNS:
    print("✅", col)


# ------------------------------------------------------------
# 11. Verify March date range
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("DATE RANGE CHECK")
print("=" * 75)

march_min = march_daily["report_date"].min()
march_max = march_daily["report_date"].max()

april_min = april_daily["report_date"].min()
april_max = april_daily["report_date"].max()

print("March:", march_min, "→", march_max)
print("April:", april_min, "→", april_max)


# ------------------------------------------------------------
# 12. Verify that dates are actually in the intended windows
# ------------------------------------------------------------

march_start = pd.Timestamp("2026-03-01")
march_end = pd.Timestamp("2026-03-31")

april_start = pd.Timestamp("2026-04-01")
april_end = pd.Timestamp("2026-04-30")


if march_min < march_start or march_max > march_end:
    raise ValueError(
        "March data contains dates outside March 2026."
    )

if april_min < april_start or april_max > april_end:
    raise ValueError(
        "April data contains dates outside April 2026."
    )


print("✅ March dates are within March 2026.")
print("✅ April dates are within April 2026.")


# ------------------------------------------------------------
# 13. Basic data-quality checks
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("BASIC DATA QUALITY CHECK")
print("=" * 75)

print(
    "March rows:",
    len(march_daily)
)

print(
    "April rows:",
    len(april_daily)
)

print(
    "March unique clients:",
    march_daily["client_hash_id"].nunique()
)

print(
    "March unique content:",
    march_daily["content_hash_id"].nunique()
)

print(
    "April unique clients:",
    april_daily["client_hash_id"].nunique()
)

print(
    "April unique content:",
    april_daily["content_hash_id"].nunique()
)


# ------------------------------------------------------------
# 14. Check missing values in important signals
# ------------------------------------------------------------

print("\nMissing values — March:")

print(
    march_daily[
        REQUIRED_COLUMNS
    ].isna().sum()
)

print("\nMissing values — April:")

print(
    april_daily[
        REQUIRED_COLUMNS
    ].isna().sum()
)


# ------------------------------------------------------------
# 15. Final status
# ------------------------------------------------------------

print("\n" + "=" * 75)
print("SECTION 2 DATA LOAD COMPLETE")
print("=" * 75)

print("✅ Hugging Face authentication works.")
print("✅ March 2026 feature data loaded.")
print("✅ April 2026 future data loaded.")
print("✅ Required columns verified.")
print("✅ Date ranges verified.")
print("✅ Basic data-quality checks completed.")

print("\nReady to continue with Section 2.")

WEEK 5 — SECTION 2 DATA LOAD
LANE 2: REFRESH / CONTENT OPPORTUNITY SCORING
✅ Hugging Face token loaded successfully.
✅ Connected to Hugging Face.
Dataset: datasets/FlyRank/internship-warehouse
Fact table: datasets/FlyRank/internship-warehouse/fact_content_daily_performance

PARTITION CHECK
March parquet files: 1
April parquet files: 1
✅ March 2026 partition found.
✅ April 2026 partition found.

Example March files:
  datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet

Example April files:
  datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet

COLUMNS USED FOR SECTION 2
• report_date
• client_hash_id
• content_hash_id
• gsc_impressions
• gsc_clicks
• gsc_avg_position

READING MARCH 2026
Loading March file 1/1...

✅ March data loaded.
March shape: (9841378, 6)

READING APRIL 2026
Loading April file 1/1...

✅ April data loaded.
April shape: (10424730, 6)

REQUIRED COLUMN CHECK
✅ report_date
✅ clie

In [6]:
# ============================================================
# WEEK 5 — SECTION 2
# SPLIT DESIGN + OBSERVED FUTURE TARGET
#
# Lane 2: Refresh / Content Opportunity Scoring
# ============================================================

import os
import numpy as np
import pandas as pd

print("=" * 75)
print("WEEK 5 — SECTION 2: SPLIT DESIGN + OBSERVED TARGET")
print("=" * 75)


# ============================================================
# 1. CHECK WEEK-4 BASELINE
# ============================================================

print("\n" + "=" * 75)
print("1. WEEK-4 BASELINE CHECK")
print("=" * 75)

if "baseline_df" not in globals():

    possible_paths = [
        "/content/baseline_action_score.csv",
        "/content/work/outputs/baseline_action_score.csv",
        "baseline_action_score.csv",
        "work/outputs/baseline_action_score.csv"
    ]

    baseline_path = None

    for path in possible_paths:
        if os.path.exists(path):
            baseline_path = path
            break

    if baseline_path is None:
        raise FileNotFoundError(
            "Week-4 baseline_action_score.csv was not found."
        )

    baseline_df = pd.read_csv(baseline_path)

print("✅ Week-4 baseline loaded.")
print("Baseline rows:", len(baseline_df))


# ============================================================
# 2. STANDARDIZE BASELINE DATE
# ============================================================

baseline_df["report_date"] = pd.to_datetime(
    baseline_df["report_date"],
    errors="coerce"
)

march_baseline = baseline_df[
    (baseline_df["report_date"] >= pd.Timestamp("2026-03-01")) &
    (baseline_df["report_date"] <= pd.Timestamp("2026-03-31"))
].copy()

print("March baseline rows:", len(march_baseline))

if len(march_baseline) == 0:
    raise ValueError(
        "No March 2026 rows exist in the Week-4 baseline."
    )

print("✅ March 2026 baseline confirmed.")


# ============================================================
# 3. CHECK FOR ALREADY-LOADED MARCH / APRIL DATA
# ============================================================

print("\n" + "=" * 75)
print("2. WAREHOUSE DATA CHECK")
print("=" * 75)

required_columns = {
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
}


def is_valid_dataframe(obj):
    """
    Check whether an object is a pandas dataframe
    containing the required warehouse columns.
    """
    return (
        isinstance(obj, pd.DataFrame)
        and required_columns.issubset(set(obj.columns))
    )


# ------------------------------------------------------------
# Case A: march_daily and april_daily already exist
# ------------------------------------------------------------

if (
    "march_daily" in globals()
    and "april_daily" in globals()
    and is_valid_dataframe(march_daily)
    and is_valid_dataframe(april_daily)
):

    print("✅ Found march_daily.")
    print("✅ Found april_daily.")

    march_daily = march_daily.copy()
    april_daily = april_daily.copy()


# ------------------------------------------------------------
# Case B: a single daily dataframe exists
# ------------------------------------------------------------

elif "daily_df" in globals() and is_valid_dataframe(daily_df):

    print("✅ Found daily_df.")

    daily_df = daily_df.copy()

    daily_df["report_date"] = pd.to_datetime(
        daily_df["report_date"],
        errors="coerce"
    )

    march_daily = daily_df[
        (daily_df["report_date"] >= "2026-03-01") &
        (daily_df["report_date"] <= "2026-03-31")
    ].copy()

    april_daily = daily_df[
        (daily_df["report_date"] >= "2026-04-01") &
        (daily_df["report_date"] <= "2026-04-30")
    ].copy()


# ------------------------------------------------------------
# Case C: another dataframe name exists
# ------------------------------------------------------------

else:

    possible_dataframe_names = [
        "df_daily",
        "warehouse_df",
        "fact_df",
        "df"
    ]

    found_dataframe = None
    found_name = None

    for name in possible_dataframe_names:

        if name in globals():

            candidate = globals()[name]

            if is_valid_dataframe(candidate):

                found_dataframe = candidate.copy()
                found_name = name
                break

    if found_dataframe is not None:

        print(
            f"✅ Found warehouse dataframe: {found_name}"
        )

        found_dataframe["report_date"] = pd.to_datetime(
            found_dataframe["report_date"],
            errors="coerce"
        )

        march_daily = found_dataframe[
            (found_dataframe["report_date"] >= "2026-03-01") &
            (found_dataframe["report_date"] <= "2026-03-31")
        ].copy()

        april_daily = found_dataframe[
            (found_dataframe["report_date"] >= "2026-04-01") &
            (found_dataframe["report_date"] <= "2026-04-30")
        ].copy()

    else:

        raise RuntimeError(
            """
SECTION 2 CANNOT CONTINUE YET.

The Week-4 baseline is loaded, but the daily warehouse
data has not been loaded into this notebook.

Required warehouse columns:

report_date
client_hash_id
content_hash_id
gsc_impressions
gsc_clicks
gsc_avg_position

You need to run the warehouse-data loading cell BEFORE
running Section 2.

Do NOT use baseline_score, reason_code, or action_label
as the future target.
"""
        )


# ============================================================
# 4. STANDARDIZE DATES
# ============================================================

print("\n" + "=" * 75)
print("3. DATE STANDARDIZATION")
print("=" * 75)

march_daily["report_date"] = pd.to_datetime(
    march_daily["report_date"],
    errors="coerce"
)

april_daily["report_date"] = pd.to_datetime(
    april_daily["report_date"],
    errors="coerce"
)

print(
    "March:",
    march_daily["report_date"].min(),
    "→",
    march_daily["report_date"].max()
)

print(
    "April:",
    april_daily["report_date"].min(),
    "→",
    april_daily["report_date"].max()
)


# ============================================================
# 5. KEEP ONLY THE INTENDED WINDOWS
# ============================================================

march_daily = march_daily[
    (march_daily["report_date"] >= "2026-03-01") &
    (march_daily["report_date"] <= "2026-03-31")
].copy()

april_daily = april_daily[
    (april_daily["report_date"] >= "2026-04-01") &
    (april_daily["report_date"] <= "2026-04-30")
].copy()


print("\nMarch rows:", len(march_daily))
print("April rows:", len(april_daily))


if len(march_daily) == 0:
    raise ValueError(
        "March warehouse data is empty."
    )

if len(april_daily) == 0:
    raise ValueError(
        "April warehouse data is empty."
    )

print("✅ Both time windows contain data.")


# ============================================================
# 6. VERIFY WAREHOUSE GRAIN
# ============================================================

print("\n" + "=" * 75)
print("4. GRAIN CHECK")
print("=" * 75)

grain = [
    "report_date",
    "client_hash_id",
    "content_hash_id"
]

duplicate_march = (
    march_daily
    .groupby(grain)
    .size()
    .reset_index(name="n")
)

duplicate_march = duplicate_march[
    duplicate_march["n"] > 1
]

print(
    "March duplicate grain rows:",
    len(duplicate_march)
)

if len(duplicate_march) > 0:
    print(
        "⚠️ Duplicate daily grain rows detected."
    )
else:
    print(
        "✅ March grain check passed:"
        "\n   one row = one report_date × client × content"
    )


# ============================================================
# 7. BUILD MARCH FEATURE TABLE
# ============================================================

print("\n" + "=" * 75)
print("5. MARCH FEATURE WINDOW")
print("=" * 75)

entity_grain = [
    "client_hash_id",
    "content_hash_id"
]

march_features = (
    march_daily
    .groupby(entity_grain, as_index=False)
    .agg(
        march_impressions=(
            "gsc_impressions",
            "sum"
        ),
        march_clicks=(
            "gsc_clicks",
            "sum"
        ),
        march_avg_position=(
            "gsc_avg_position",
            "mean"
        )
    )
)

print(
    "March feature rows:",
    len(march_features)
)

print("\nExample:")
display(
    march_features.head()
)


# ============================================================
# 8. BUILD APRIL OUTCOME TABLE
# ============================================================

print("\n" + "=" * 75)
print("6. APRIL FUTURE OUTCOME WINDOW")
print("=" * 75)

april_outcomes = (
    april_daily
    .groupby(entity_grain, as_index=False)
    .agg(
        april_impressions=(
            "gsc_impressions",
            "sum"
        ),
        april_clicks=(
            "gsc_clicks",
            "sum"
        ),
        april_avg_position=(
            "gsc_avg_position",
            "mean"
        )
    )
)

print(
    "April outcome rows:",
    len(april_outcomes)
)

display(
    april_outcomes.head()
)


# ============================================================
# 9. JOIN MARCH FEATURES TO APRIL OUTCOMES
# ============================================================

print("\n" + "=" * 75)
print("7. MARCH → APRIL JOIN")
print("=" * 75)

model_df = march_features.merge(
    april_outcomes,
    on=entity_grain,
    how="inner"
)

print(
    "Rows with both March features and April outcomes:",
    len(model_df)
)

print(
    "March-only rows:",
    len(march_features)
)

print(
    "April-only rows:",
    len(april_outcomes)
)

print(
    "Joined rows:",
    len(model_df)
)


# ============================================================
# 10. APPLY MINIMUM MARCH VOLUME FILTER
# ============================================================

print("\n" + "=" * 75)
print("8. ELIGIBILITY FILTER")
print("=" * 75)

MIN_IMPRESSIONS = 100
MIN_CLICKS = 5

eligible_df = model_df[
    (model_df["march_impressions"] >= MIN_IMPRESSIONS)
    &
    (model_df["march_clicks"] >= MIN_CLICKS)
].copy()

print(
    "Minimum March impressions:",
    MIN_IMPRESSIONS
)

print(
    "Minimum March clicks:",
    MIN_CLICKS
)

print(
    "Eligible rows:",
    len(eligible_df)
)


# ============================================================
# 11. CREATE FUTURE OUTCOME
# ============================================================

print("\n" + "=" * 75)
print("9. OBSERVED FUTURE TARGET")
print("=" * 75)

eligible_df["click_change_pct"] = (
    (
        eligible_df["april_clicks"]
        - eligible_df["march_clicks"]
    )
    /
    eligible_df["march_clicks"]
    * 100
)


eligible_df["future_decline"] = (
    eligible_df["click_change_pct"] <= -20
).astype(int)


print(
    "Target definition:"
)

print(
    "future_decline = 1 when April clicks are "
    "at least 20% lower than March clicks."
)


print("\nTarget distribution:")

target_counts = (
    eligible_df["future_decline"]
    .value_counts()
    .sort_index()
    .rename_axis("future_decline")
    .to_frame("n")
)

display(target_counts)


target_rate = (
    eligible_df["future_decline"]
    .mean()
)

print(
    f"\nFuture-decline rate: {target_rate:.2%}"
)


# ============================================================
# 12. ATTACH WEEK-4 BASELINE
# ============================================================

print("\n" + "=" * 75)
print("10. WEEK-4 BASELINE JOIN")
print("=" * 75)

baseline_required = [
    "client_hash_id",
    "content_hash_id",
    "baseline_score",
    "reason_code",
    "action_label"
]

missing_baseline_columns = [
    c for c in baseline_required
    if c not in march_baseline.columns
]

if missing_baseline_columns:

    raise ValueError(
        "Week-4 baseline is missing required columns: "
        + str(missing_baseline_columns)
    )


baseline_for_join = march_baseline[
    baseline_required
].copy()


# Keep one baseline decision per client × content.
baseline_for_join = (
    baseline_for_join
    .sort_values(
        "baseline_score",
        ascending=False
    )
    .drop_duplicates(
        subset=entity_grain,
        keep="first"
    )
)


comparison_df = eligible_df.merge(
    baseline_for_join,
    on=entity_grain,
    how="inner"
)


print(
    "Rows available for model + baseline comparison:",
    len(comparison_df)
)


# ============================================================
# 13. LEAKAGE CHECK
# ============================================================

print("\n" + "=" * 75)
print("11. LEAKAGE CHECK")
print("=" * 75)


feature_columns = [
    "march_impressions",
    "march_clicks",
    "march_avg_position"
]


forbidden_columns = [
    "baseline_score",
    "reason_code",
    "action_label",
    "april_impressions",
    "april_clicks",
    "april_avg_position",
    "click_change_pct",
    "future_decline"
]


leakage_found = [
    col
    for col in feature_columns
    if col in forbidden_columns
]


if leakage_found:

    raise ValueError(
        "❌ Leakage detected: "
        + str(leakage_found)
    )

else:

    print(
        "✅ March feature columns contain no future target."
    )

    print(
        "✅ Week-4 baseline outputs are not model features."
    )


# ============================================================
# 14. FINAL SECTION 2 OBJECTS
# ============================================================

print("\n" + "=" * 75)
print("SECTION 2 COMPLETE")
print("=" * 75)

print(
    "Decision window : March 2026"
)

print(
    "Outcome window  : April 2026"
)

print(
    "Feature grain   : client × content"
)

print(
    "Target          : future_decline"
)

print(
    "Features        :",
    feature_columns
)

print(
    "Baseline        : Week-4 baseline_score"
)

print(
    "Baseline as ML feature : NO"
)

print(
    "April as ML feature    : NO"
)

print(
    "\n✅ Section 2 is ready for Section 3."
)

WEEK 5 — SECTION 2: SPLIT DESIGN + OBSERVED TARGET

1. WEEK-4 BASELINE CHECK
✅ Week-4 baseline loaded.
Baseline rows: 7928488
March baseline rows: 7928488
✅ March 2026 baseline confirmed.

2. WAREHOUSE DATA CHECK
✅ Found march_daily.
✅ Found april_daily.

3. DATE STANDARDIZATION
March: 2026-03-01 00:00:00 → 2026-03-31 00:00:00
April: 2026-04-01 00:00:00 → 2026-04-30 00:00:00

March rows: 9841378
April rows: 10424730
✅ Both time windows contain data.

4. GRAIN CHECK
March duplicate grain rows: 0
✅ March grain check passed:
   one row = one report_date × client × content

5. MARCH FEATURE WINDOW
March feature rows: 331437

Example:


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_avg_position
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,0,0,NaN
1,client_0797ff3a1fc9a6a5,content_0236ef736698e17c,0,0,NaN
2,client_0797ff3a1fc9a6a5,content_025f6cfd3c298870,0,0,NaN
3,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9.0
4,client_0797ff3a1fc9a6a5,content_02752c6c1c60161f,0,0,NaN



6. APRIL FUTURE OUTCOME WINDOW
April outcome rows: 362172


,client_hash_id,content_hash_id,april_impressions,april_clicks,april_avg_position
0,client_06d356715a8ff3b6,content_0059a4d4195810c9,873,2,8.041344
1,client_06d356715a8ff3b6,content_005b6b7f7b8dda7f,634,1,9.323939
2,client_06d356715a8ff3b6,content_0153b7dedc3fc40d,640,2,7.919196
3,client_06d356715a8ff3b6,content_0241f6a890063db0,85,1,3.573096
4,client_06d356715a8ff3b6,content_045f673b3d3c18a4,153,2,3.980769



7. MARCH → APRIL JOIN
Rows with both March features and April outcomes: 331436
March-only rows: 331437
April-only rows: 362172
Joined rows: 331436

8. ELIGIBILITY FILTER
Minimum March impressions: 100
Minimum March clicks: 5
Eligible rows: 28769

9. OBSERVED FUTURE TARGET
Target definition:
future_decline = 1 when April clicks are at least 20% lower than March clicks.

Target distribution:


,n
future_decline,
0,12658
1,16111



Future-decline rate: 56.00%

10. WEEK-4 BASELINE JOIN
Rows available for model + baseline comparison: 27513

11. LEAKAGE CHECK
✅ March feature columns contain no future target.
✅ Week-4 baseline outputs are not model features.

SECTION 2 COMPLETE
Decision window : March 2026
Outcome window  : April 2026
Feature grain   : client × content
Target          : future_decline
Features        : ['march_impressions', 'march_clicks', 'march_avg_position']
Baseline        : Week-4 baseline_score
Baseline as ML feature : NO
April as ML feature    : NO

✅ Section 2 is ready for Section 3.


## 3. Train the model and compare with the Week-4 baseline

### Objective

The purpose of this section is to test whether a learned model can rank future-declining content items more effectively than the transparent Week-4 baseline.

The Week-4 baseline remains the benchmark. I will not assume that a more complex model is better simply because it is a machine-learning model.

### Target

The model predicts:

`future_decline`

where:

- `1` means April clicks were at least 20% lower than March clicks;
- `0` means they did not meet that decline threshold.

This target is based on observed April behavior and is not copied from the Week-4 action label.

### Features

The initial model uses only March information:

- `march_impressions`
- `march_clicks`
- `march_avg_position`

I deliberately exclude:

- `baseline_score`
- `reason_code`
- `action_label`
- April metrics
- `click_change_pct`
- `future_decline`
- client/content identifiers

This prevents the model from learning the Week-4 rule or seeing the future outcome.

### Models

I will compare two learned approaches:

1. Logistic Regression — a simple, interpretable linear reference model.
2. Random Forest — a more flexible nonlinear model that can learn interactions between the signals.

The goal is not to reward complexity. If the simpler model performs similarly, that is useful evidence.

### Validation design

I will use a grouped train/test split by `client_hash_id`.

This prevents content items from the same client from being split across training and test sets.

The test set is held out before model fitting.

Both the learned model and the Week-4 baseline will be evaluated on exactly the same test rows.

### Ranking evaluation

Because Lane 2 is a prioritization problem, the model output will be treated as a ranking score.

I will compare:

- Precision@100
- Precision@500
- Precision@1000
- Average Precision

The same future-decline target will be used for both the ML model and the Week-4 baseline.

### Interpretation

A model is useful only if it improves the decision.

If Random Forest is only slightly better than Logistic Regression or the Week-4 baseline, I will report that honestly rather than selecting the more complex model simply because it is more sophisticated.

The final interpretation will focus on:

- whether ranking quality improved;
- which model performed best;
- what the errors look like;
- whether the improvement is large enough to justify additional complexity.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# WEEK 5 — SECTION 3
# TRAIN + COMPARE AGAINST WEEK-4 BASELINE
#
# Lane 2: Refresh / Content Opportunity Scoring
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    classification_report
)

print("=" * 75)
print("WEEK 5 — SECTION 3: TRAIN + COMPARE")
print("=" * 75)


# ============================================================
# 1. VERIFY SECTION 2 OUTPUT
# ============================================================

print("\n" + "=" * 75)
print("1. SECTION 2 OBJECT CHECK")
print("=" * 75)

required_objects = [
    "comparison_df"
]

for obj in required_objects:

    if obj not in globals():

        raise RuntimeError(
            f"{obj} is not available. "
            "Run Section 2 first."
        )

    print(f"✅ {obj} found.")


# ============================================================
# 2. DEFINE FEATURES AND TARGET
# ============================================================

features = [
    "march_impressions",
    "march_clicks",
    "march_avg_position"
]

target = "future_decline"

required_columns = (
    features
    + [
        target,
        "client_hash_id",
        "content_hash_id",
        "baseline_score"
    ]
)

missing_columns = [
    col
    for col in required_columns
    if col not in comparison_df.columns
]

if missing_columns:

    raise ValueError(
        "Missing required columns: "
        + str(missing_columns)
    )


model_data = comparison_df[
    required_columns
].copy()


print("\n" + "=" * 75)
print("2. MODEL DATA")
print("=" * 75)

print("Rows:", len(model_data))
print("Features:", features)
print("Target:", target)


# ============================================================
# 3. CLEAN NUMERIC FEATURES
# ============================================================

for col in features:

    model_data[col] = pd.to_numeric(
        model_data[col],
        errors="coerce"
    )


model_data[target] = pd.to_numeric(
    model_data[target],
    errors="coerce"
)


model_data = model_data.dropna(
    subset=[target]
).copy()


print("\nTarget distribution:")

display(
    model_data[target]
    .value_counts()
    .sort_index()
    .rename_axis(target)
    .to_frame("n")
)


# ============================================================
# 4. GROUPED TRAIN / TEST SPLIT
# ============================================================

print("\n" + "=" * 75)
print("3. GROUPED TRAIN / TEST SPLIT")
print("=" * 75)

groups = model_data["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_data,
        model_data[target],
        groups=groups
    )
)


train_df = model_data.iloc[
    train_idx
].copy()

test_df = model_data.iloc[
    test_idx
].copy()


print(
    "Training rows:",
    len(train_df)
)

print(
    "Test rows:",
    len(test_df)
)

print(
    "Training clients:",
    train_df["client_hash_id"].nunique()
)

print(
    "Test clients:",
    test_df["client_hash_id"].nunique()
)


# ============================================================
# 5. VERIFY NO CLIENT OVERLAP
# ============================================================

train_clients = set(
    train_df["client_hash_id"]
)

test_clients = set(
    test_df["client_hash_id"]
)

overlap = train_clients.intersection(
    test_clients
)

print(
    "\nClient overlap between train and test:",
    len(overlap)
)

if len(overlap) != 0:

    raise ValueError(
        "Client leakage detected between train and test."
    )

else:

    print(
        "✅ No client overlap."
    )


# ============================================================
# 6. CREATE X / y
# ============================================================

X_train = train_df[
    features
]

y_train = train_df[
    target
].astype(int)


X_test = test_df[
    features
]

y_test = test_df[
    target
].astype(int)


# ============================================================
# 7. DEFINE LOGISTIC REGRESSION
# ============================================================

logistic_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        ),
        (
            "model",
            LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)


# ============================================================
# 8. DEFINE RANDOM FOREST
# ============================================================

random_forest_model = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=250,
                max_depth=8,
                min_samples_leaf=20,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)


# ============================================================
# 9. TRAIN LOGISTIC REGRESSION
# ============================================================

print("\n" + "=" * 75)
print("4. TRAINING LOGISTIC REGRESSION")
print("=" * 75)

logistic_model.fit(
    X_train,
    y_train
)

logistic_probability = (
    logistic_model
    .predict_proba(X_test)[:, 1]
)

print(
    "✅ Logistic Regression trained."
)


# ============================================================
# 10. TRAIN RANDOM FOREST
# ============================================================

print("\n" + "=" * 75)
print("5. TRAINING RANDOM FOREST")
print("=" * 75)

random_forest_model.fit(
    X_train,
    y_train
)

rf_probability = (
    random_forest_model
    .predict_proba(X_test)[:, 1]
)

print(
    "✅ Random Forest trained."
)


# ============================================================
# 11. BASELINE SCORE
# ============================================================

baseline_probability = pd.to_numeric(
    test_df["baseline_score"],
    errors="coerce"
)

# Baseline should be ranked highest score first.

print("\n" + "=" * 75)
print("6. WEEK-4 BASELINE")
print("=" * 75)

print(
    "Baseline scores available:",
    baseline_probability.notna().sum()
)

if baseline_probability.isna().any():

    print(
        "⚠️ Some baseline scores are missing."
    )


# ============================================================
# 12. CREATE RANKING DATAFRAME
# ============================================================

ranking_df = test_df[
    [
        "client_hash_id",
        "content_hash_id",
        target,
        "baseline_score"
    ]
].copy()


ranking_df["logistic_score"] = (
    logistic_probability
)

ranking_df["rf_score"] = (
    rf_probability
)


print("\nRanking dataframe created.")

display(
    ranking_df.head()
)


# ============================================================
# 13. PRECISION@K FUNCTION
# ============================================================

def precision_at_k(
    y_true,
    scores,
    k
):

    y_true = np.asarray(
        y_true
    )

    scores = np.asarray(
        scores
    )

    valid = np.isfinite(scores)

    y_true = y_true[valid]
    scores = scores[valid]

    if len(y_true) == 0:
        return np.nan

    k = min(
        k,
        len(y_true)
    )

    order = np.argsort(
        -scores
    )

    top_k = order[:k]

    return float(
        y_true[top_k].mean()
    )


# ============================================================
# 14. AVERAGE PRECISION FUNCTION
# ============================================================

def safe_average_precision(
    y_true,
    scores
):

    y_true = np.asarray(
        y_true
    )

    scores = np.asarray(
        scores
    )

    valid = np.isfinite(scores)

    y_true = y_true[valid]
    scores = scores[valid]

    if len(np.unique(y_true)) < 2:

        return np.nan

    return float(
        average_precision_score(
            y_true,
            scores
        )
    )


# ============================================================
# 15. EVALUATE ALL THREE SYSTEMS
# ============================================================

print("\n" + "=" * 75)
print("7. RANKING EVALUATION")
print("=" * 75)

K_VALUES = [
    100,
    500,
    1000
]

results = []


for name, scores in [

    (
        "Week-4 Baseline",
        ranking_df["baseline_score"].values
    ),

    (
        "Logistic Regression",
        ranking_df["logistic_score"].values
    ),

    (
        "Random Forest",
        ranking_df["rf_score"].values
    )

]:

    row = {
        "system": name
    }

    for k in K_VALUES:

        row[
            f"precision_at_{k}"
        ] = precision_at_k(
            ranking_df[target].values,
            scores,
            k
        )

    row[
        "average_precision"
    ] = safe_average_precision(
        ranking_df[target].values,
        scores
    )

    results.append(
        row
    )


results_df = pd.DataFrame(
    results
)


# ============================================================
# 16. DISPLAY COMPARISON
# ============================================================

print("\n" + "=" * 75)
print("MODEL VS BASELINE")
print("=" * 75)

display(
    results_df.round(4)
)


# ============================================================
# 17. FIND BEST SYSTEM
# ============================================================

best_system = (
    results_df
    .sort_values(
        "average_precision",
        ascending=False
    )
    .iloc[0]
)


print(
    "\nBest system by Average Precision:"
)

print(
    best_system["system"]
)

print(
    "Average Precision:",
    round(
        best_system["average_precision"],
        4
    )
)


# ============================================================
# 18. LOGISTIC REGRESSION CLASSIFICATION REPORT
# ============================================================

logistic_predictions = (
    logistic_probability >= 0.50
).astype(int)


print("\n" + "=" * 75)
print("8. LOGISTIC REGRESSION CLASSIFICATION REPORT")
print("=" * 75)

print(
    classification_report(
        y_test,
        logistic_predictions,
        digits=3
    )
)


# ============================================================
# 19. RANDOM FOREST CLASSIFICATION REPORT
# ============================================================

rf_predictions = (
    rf_probability >= 0.50
).astype(int)


print("\n" + "=" * 75)
print("9. RANDOM FOREST CLASSIFICATION REPORT")
print("=" * 75)

print(
    classification_report(
        y_test,
        rf_predictions,
        digits=3
    )
)


# ============================================================
# 20. RANDOM FOREST FEATURE IMPORTANCE
# ============================================================

print("\n" + "=" * 75)
print("10. RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 75)

rf_estimator = (
    random_forest_model
    .named_steps["model"]
)

importance_df = pd.DataFrame({
    "feature": features,
    "importance": rf_estimator.feature_importances_
})

importance_df = (
    importance_df
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    importance_df.round(4)
)


# ============================================================
# 21. SAVE SECTION 3 RESULTS
# ============================================================

results_df.to_csv(
    "w05_model_vs_baseline_metrics.csv",
    index=False
)


ranking_df.to_csv(
    "w05_test_rankings.csv",
    index=False
)


# ============================================================
# 22. FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("SECTION 3 COMPLETE")
print("=" * 75)

print(
    "✅ Grouped train/test split completed."
)

print(
    "✅ Logistic Regression trained."
)

print(
    "✅ Random Forest trained."
)

print(
    "✅ Week-4 baseline evaluated on the same test rows."
)

print(
    "✅ Precision@K calculated."
)

print(
    "✅ Average Precision calculated."
)

print(
    "✅ Feature importance calculated."
)

print(
    "✅ Model comparison saved."
)

print(
    "\nNext: SECTION 4 — Errors and interpretation."
)

WEEK 5 — SECTION 3: TRAIN + COMPARE

1. SECTION 2 OBJECT CHECK
✅ comparison_df found.

2. MODEL DATA
Rows: 27513
Features: ['march_impressions', 'march_clicks', 'march_avg_position']
Target: future_decline

Target distribution:


,n
future_decline,
0,12151
1,15362



3. GROUPED TRAIN / TEST SPLIT
Training rows: 14245
Test rows: 13268
Training clients: 25
Test clients: 7

Client overlap between train and test: 0
✅ No client overlap.

4. TRAINING LOGISTIC REGRESSION
✅ Logistic Regression trained.

5. TRAINING RANDOM FOREST
✅ Random Forest trained.

6. WEEK-4 BASELINE
Baseline scores available: 13268

Ranking dataframe created.


,client_hash_id,content_hash_id,future_decline,baseline_score,logistic_score,rf_score
4279,client_23a62021009f63c4,content_000ff3abcd61db63,1,0.024945,0.411055,0.393134
4280,client_23a62021009f63c4,content_0015df53ad201ad5,0,0.044234,0.326435,0.326548
4281,client_23a62021009f63c4,content_0015e9e481380d7d,1,0.021239,0.412485,0.509534
4282,client_23a62021009f63c4,content_0018b50e392faeb8,0,0.036727,0.251140,0.327171
4283,client_23a62021009f63c4,content_001ccd954baa493f,0,0.030614,0.367840,0.507994



7. RANKING EVALUATION

MODEL VS BASELINE


,system,precision_at_100,precision_at_500,precision_at_1000,average_precision
0,Week-4 Baseline,0.58,0.546,0.528,0.5593
1,Logistic Regression,0.51,0.588,0.591,0.5793
2,Random Forest,0.49,0.614,0.611,0.5932



Best system by Average Precision:
Random Forest
Average Precision: 0.5932

8. LOGISTIC REGRESSION CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0      0.446     0.518     0.479      5759
           1      0.578     0.506     0.540      7509

    accuracy                          0.511     13268
   macro avg      0.512     0.512     0.509     13268
weighted avg      0.520     0.511     0.513     13268


9. RANDOM FOREST CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0      0.482     0.468     0.475      5759
           1      0.601     0.615     0.608      7509

    accuracy                          0.551     13268
   macro avg      0.542     0.541     0.541     13268
weighted avg      0.549     0.551     0.550     13268


10. RANDOM FOREST FEATURE IMPORTANCE


,feature,importance
0,march_avg_position,0.3730
1,march_clicks,0.3334
2,march_impressions,0.2936



SECTION 3 COMPLETE
✅ Grouped train/test split completed.
✅ Logistic Regression trained.
✅ Random Forest trained.
✅ Week-4 baseline evaluated on the same test rows.
✅ Precision@K calculated.
✅ Average Precision calculated.
✅ Feature importance calculated.
✅ Model comparison saved.

Next: SECTION 4 — Errors and interpretation.


## 4. Errors and interpretation

### What I am inspecting

The model is not useful simply because it produces a score. I need to understand where the ranking succeeds and where it fails.

For this Lane 2 task, I inspect:

- false positives;
- false negatives;
- high-confidence mistakes;
- differences between the Random Forest ranking and the Week-4 baseline.

### False positives

A false positive is a page that the model ranks as likely to experience future decline, but the observed April outcome does not meet the `future_decline` definition.

These cases matter because they could cause a reviewer to spend time investigating a page that did not subsequently decline.

### False negatives

A false negative is a page that the model ranks as relatively unlikely to experience future decline, but the page actually meets the future-decline definition.

These cases matter because they represent pages the prioritization system could miss.

### Interpretation

The model should be treated as a prioritization tool, not as proof that a page needs a refresh.

A prediction can be wrong because the three available March signals do not contain enough information to explain the later outcome. Other factors may affect future clicks, including changes in search demand, competition, content changes, seasonality, or measurement conditions.

### Baseline versus model

The Week-4 baseline is strongest at the very top of the queue in this experiment:

- Baseline Precision@100 = 0.580
- Random Forest Precision@100 = 0.490

However, Random Forest performs better at larger review budgets:

- Random Forest Precision@500 = 0.614 versus baseline 0.546
- Random Forest Precision@1000 = 0.611 versus baseline 0.528

Random Forest also has higher Average Precision:

- Week-4 baseline = 0.5593
- Random Forest = 0.5932

Therefore, the result is not a simple "model wins everywhere" story. The learned model improves the broader ranking, while the existing rule performs better at the very top 100 positions.

### Main limitation

The model uses only three March GSC signals. This limits what it can learn about future content performance.

The target is also a proxy for subsequent click decline, not a causal label for whether a page would benefit from a refresh.

The model should therefore support human review rather than automatically trigger content changes.

### What I learned

The main lesson is that model evaluation depends on the decision being supported.

A model can have a better overall ranking metric while still performing worse at the smallest review budget. Therefore, I should not select a model using one metric alone.

The errors also show why the model should be treated as decision support rather than an automatic content-action system.444

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# WEEK 5 — SECTION 4
# ERRORS AND INTERPRETATION
#
# Lane 2: Refresh / Content Opportunity Scoring
# ============================================================

import numpy as np
import pandas as pd

print("=" * 75)
print("WEEK 5 — SECTION 4: ERRORS AND INTERPRETATION")
print("=" * 75)


# ============================================================
# 1. VERIFY SECTION 3 OUTPUT
# ============================================================

print("\n" + "=" * 75)
print("1. SECTION 3 OBJECT CHECK")
print("=" * 75)

required_objects = [
    "ranking_df",
    "results_df",
    "importance_df"
]

for obj in required_objects:

    if obj not in globals():
        raise RuntimeError(
            f"{obj} is not available. "
            "Run Section 3 first."
        )

    print(f"✅ {obj} found.")


# ============================================================
# 2. RECREATE TEST TARGET
# ============================================================

y_true = ranking_df[
    "future_decline"
].astype(int).values

rf_scores = ranking_df[
    "rf_score"
].astype(float).values

baseline_scores = ranking_df[
    "baseline_score"
].astype(float).values


# ============================================================
# 3. CREATE RF PREDICTIONS
# ============================================================

# 0.50 is used only to define classification errors.
# Ranking evaluation was already performed using the
# continuous RF probability.

rf_pred = (
    rf_scores >= 0.50
).astype(int)


ranking_df["rf_prediction"] = rf_pred


# ============================================================
# 4. ERROR TYPES
# ============================================================

ranking_df["error_type"] = "Correct"

ranking_df.loc[
    (ranking_df["rf_prediction"] == 1)
    & (ranking_df["future_decline"] == 0),
    "error_type"
] = "False Positive"


ranking_df.loc[
    (ranking_df["rf_prediction"] == 0)
    & (ranking_df["future_decline"] == 1),
    "error_type"
] = "False Negative"


# ============================================================
# 5. ERROR SUMMARY
# ============================================================

print("\n" + "=" * 75)
print("2. RANDOM FOREST ERROR SUMMARY")
print("=" * 75)

error_summary = (
    ranking_df["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .to_frame("n")
)

error_summary["percentage"] = (
    error_summary["n"]
    / len(ranking_df)
    * 100
)

display(
    error_summary.round(2)
)


# ============================================================
# 6. FALSE POSITIVES
# ============================================================

print("\n" + "=" * 75)
print("3. FALSE POSITIVES")
print("=" * 75)

false_positive_df = (
    ranking_df[
        ranking_df["error_type"]
        == "False Positive"
    ]
    .sort_values(
        "rf_score",
        ascending=False
    )
    .copy()
)

print(
    "False positives:",
    len(false_positive_df)
)

display(
    false_positive_df[
        [
            "client_hash_id",
            "content_hash_id",
            "future_decline",
            "rf_score",
            "baseline_score"
        ]
    ].head(10)
)


# ============================================================
# 7. FALSE NEGATIVES
# ============================================================

print("\n" + "=" * 75)
print("4. FALSE NEGATIVES")
print("=" * 75)

false_negative_df = (
    ranking_df[
        ranking_df["error_type"]
        == "False Negative"
    ]
    .sort_values(
        "rf_score",
        ascending=True
    )
    .copy()
)

print(
    "False negatives:",
    len(false_negative_df)
)

display(
    false_negative_df[
        [
            "client_hash_id",
            "content_hash_id",
            "future_decline",
            "rf_score",
            "baseline_score"
        ]
    ].head(10)
)


# ============================================================
# 8. HIGH-CONFIDENCE FALSE POSITIVES
# ============================================================

print("\n" + "=" * 75)
print("5. HIGH-CONFIDENCE FALSE POSITIVES")
print("=" * 75)

high_conf_fp = (
    false_positive_df[
        false_positive_df["rf_score"] >= 0.70
    ]
    .copy()
)

print(
    "High-confidence false positives:",
    len(high_conf_fp)
)

display(
    high_conf_fp[
        [
            "client_hash_id",
            "content_hash_id",
            "rf_score",
            "baseline_score",
            "future_decline"
        ]
    ].head(10)
)


# ============================================================
# 9. HIGH-CONFIDENCE FALSE NEGATIVES
# ============================================================

print("\n" + "=" * 75)
print("6. HIGH-CONFIDENCE FALSE NEGATIVES")
print("=" * 75)

high_conf_fn = (
    false_negative_df[
        false_negative_df["rf_score"] <= 0.30
    ]
    .copy()
)

print(
    "High-confidence false negatives:",
    len(high_conf_fn)
)

display(
    high_conf_fn[
        [
            "client_hash_id",
            "content_hash_id",
            "rf_score",
            "baseline_score",
            "future_decline"
        ]
    ].head(10)
)


# ============================================================
# 10. TOP-10 RANDOM FOREST RANKING
# ============================================================

print("\n" + "=" * 75)
print("7. RANDOM FOREST TOP 10")
print("=" * 75)

rf_top10 = (
    ranking_df
    .sort_values(
        "rf_score",
        ascending=False
    )
    .head(10)
    .copy()
)

display(
    rf_top10[
        [
            "client_hash_id",
            "content_hash_id",
            "future_decline",
            "rf_score",
            "baseline_score"
        ]
    ]
)


# ============================================================
# 11. TOP-10 BASELINE RANKING
# ============================================================

print("\n" + "=" * 75)
print("8. WEEK-4 BASELINE TOP 10")
print("=" * 75)

baseline_top10 = (
    ranking_df
    .sort_values(
        "baseline_score",
        ascending=False
    )
    .head(10)
    .copy()
)

display(
    baseline_top10[
        [
            "client_hash_id",
            "content_hash_id",
            "future_decline",
            "baseline_score",
            "rf_score"
        ]
    ]
)


# ============================================================
# 12. COMPARE TOP-100 OVERLAP
# ============================================================

print("\n" + "=" * 75)
print("9. TOP-100 RANKING OVERLAP")
print("=" * 75)

rf_top100_ids = set(
    ranking_df
    .sort_values(
        "rf_score",
        ascending=False
    )
    .head(100)
    ["content_hash_id"]
)

baseline_top100_ids = set(
    ranking_df
    .sort_values(
        "baseline_score",
        ascending=False
    )
    .head(100)
    ["content_hash_id"]
)

top100_overlap = (
    len(
        rf_top100_ids.intersection(
            baseline_top100_ids
        )
    )
)

print(
    "RF top-100 pages:",
    len(rf_top100_ids)
)

print(
    "Baseline top-100 pages:",
    len(baseline_top100_ids)
)

print(
    "Overlap:",
    top100_overlap
)

print(
    "Overlap percentage:",
    f"{top100_overlap}%"
)


# ============================================================
# 13. FEATURE IMPORTANCE
# ============================================================

print("\n" + "=" * 75)
print("10. RANDOM FOREST FEATURE IMPORTANCE")
print("=" * 75)

display(
    importance_df.round(4)
)


# ============================================================
# 14. INTERPRETATION NUMBERS
# ============================================================

print("\n" + "=" * 75)
print("11. INTERPRETATION SUMMARY")
print("=" * 75)

baseline_ap = (
    results_df.loc[
        results_df["system"]
        == "Week-4 Baseline",
        "average_precision"
    ].iloc[0]
)

rf_ap = (
    results_df.loc[
        results_df["system"]
        == "Random Forest",
        "average_precision"
    ].iloc[0]
)

baseline_p100 = (
    results_df.loc[
        results_df["system"]
        == "Week-4 Baseline",
        "precision_at_100"
    ].iloc[0]
)

rf_p100 = (
    results_df.loc[
        results_df["system"]
        == "Random Forest",
        "precision_at_100"
    ].iloc[0]
)

baseline_p500 = (
    results_df.loc[
        results_df["system"]
        == "Week-4 Baseline",
        "precision_at_500"
    ].iloc[0]
)

rf_p500 = (
    results_df.loc[
        results_df["system"]
        == "Random Forest",
        "precision_at_500"
    ].iloc[0]
)

baseline_p1000 = (
    results_df.loc[
        results_df["system"]
        == "Week-4 Baseline",
        "precision_at_1000"
    ].iloc[0]
)

rf_p1000 = (
    results_df.loc[
        results_df["system"]
        == "Random Forest",
        "precision_at_1000"
    ].iloc[0]
)

print(
    f"Baseline Average Precision : {baseline_ap:.4f}"
)

print(
    f"RF Average Precision       : {rf_ap:.4f}"
)

print(
    f"Average Precision change   : {rf_ap - baseline_ap:+.4f}"
)

print()

print(
    f"Baseline Precision@100 : {baseline_p100:.3f}"
)

print(
    f"RF Precision@100       : {rf_p100:.3f}"
)

print()

print(
    f"Baseline Precision@500 : {baseline_p500:.3f}"
)

print(
    f"RF Precision@500       : {rf_p500:.3f}"
)

print()

print(
    f"Baseline Precision@1000 : {baseline_p1000:.3f}"
)

print(
    f"RF Precision@1000       : {rf_p1000:.3f}"
)


# ============================================================
# 15. SAVE ERROR TABLES
# ============================================================

false_positive_df.to_csv(
    "w05_false_positives.csv",
    index=False
)

false_negative_df.to_csv(
    "w05_false_negatives.csv",
    index=False
)

importance_df.to_csv(
    "w05_feature_importance.csv",
    index=False
)


# ============================================================
# 16. FINAL STATUS
# ============================================================

print("\n" + "=" * 75)
print("SECTION 4 COMPLETE")
print("=" * 75)

print(
    "✅ False positives identified."
)

print(
    "✅ False negatives identified."
)

print(
    "✅ High-confidence errors inspected."
)

print(
    "✅ Random Forest top-10 inspected."
)

print(
    "✅ Week-4 baseline top-10 inspected."
)

print(
    "✅ Feature importance reviewed."
)

print(
    "✅ Ranking overlap calculated."
)

print(
    "✅ Error tables saved."
)

print(
    "\nNext: SECTION 5 — Self-check."
)

WEEK 5 — SECTION 4: ERRORS AND INTERPRETATION

1. SECTION 3 OBJECT CHECK
✅ ranking_df found.
✅ results_df found.
✅ importance_df found.

2. RANDOM FOREST ERROR SUMMARY


,n,percentage
error_type,,
Correct,7310,55.09
False Positive,3066,23.11
False Negative,2892,21.80



3. FALSE POSITIVES
False positives: 3066


,client_hash_id,content_hash_id,future_decline,rf_score,baseline_score
15315,client_73cda7b4e4f265ea,content_2452cf1fb7b5364b,0,0.758337,0.011853
14482,client_73cda7b4e4f265ea,content_02c4ea0cf6fad4f7,0,0.743371,0.006184
19846,client_73cda7b4e4f265ea,content_dd6332be0b60edfc,0,0.740621,0.009103
27010,client_fef1a8f436438636,content_d4da05eca96514f7,0,0.735446,0.006425
18054,client_73cda7b4e4f265ea,content_956351595e58cef6,0,0.734309,0.006537
18647,client_73cda7b4e4f265ea,content_ae086c23a290c435,0,0.731897,0.006672
25884,client_fef1a8f436438636,content_457f11675f080d37,0,0.729871,0.007822
19692,client_73cda7b4e4f265ea,content_d7afd18d30e80c03,0,0.728531,0.006024
14604,client_73cda7b4e4f265ea,content_07b7918e6f958218,0,0.727140,0.008803
15197,client_73cda7b4e4f265ea,content_1f738927b8104015,0,0.724937,0.006472



4. FALSE NEGATIVES
False negatives: 2892


...


8. WEEK-4 BASELINE TOP 10


,client_hash_id,content_hash_id,future_decline,baseline_score,rf_score
5549,client_23a62021009f63c4,content_44f34c0a90047651,1,0.600067,0.560653
6147,client_23a62021009f63c4,content_66288edeb93b7c4f,1,0.376553,0.402955
17045,client_73cda7b4e4f265ea,content_6a9c79f55413b447,0,0.330597,0.443093
8673,client_23a62021009f63c4,content_e8a52cf3d5988c07,0,0.244319,0.396488
8623,client_23a62021009f63c4,content_e6df0936699f5b8f,1,0.239878,0.284502
8688,client_23a62021009f63c4,content_e943d753806d7af3,1,0.239401,0.377311
6447,client_23a62021009f63c4,content_74de5f247659e956,1,0.178180,0.292156
6141,client_23a62021009f63c4,content_65c75874a23fca87,1,0.173431,0.520369
5299,client_23a62021009f63c4,content_36e53e9c707674fc,0,0.167056,0.401880
5045,client_23a62021009f63c4,content_292bf485e0f2599e,1,0.166280,0.630138



9. TOP-100 RANKING OVERLAP
RF top-100 pages: 100
Baseline top-100 pages: 100
Overlap: 0
Overlap percentage: 0%

10. RANDOM FOREST FEATURE IMPORTANCE


,feature,importance
0,march_avg_position,0.3730
1,march_clicks,0.3334
2,march_impressions,0.2936



11. INTERPRETATION SUMMARY
Baseline Average Precision : 0.5593
RF Average Precision       : 0.5932
Average Precision change   : +0.0340

Baseline Precision@100 : 0.580
RF Precision@100       : 0.490

Baseline Precision@500 : 0.546
RF Precision@500       : 0.614

Baseline Precision@1000 : 0.528
RF Precision@1000       : 0.611

SECTION 4 COMPLETE
✅ False positives identified.
✅ False negatives identified.
✅ High-confidence errors inspected.
✅ Random Forest top-10 inspected.
✅ Week-4 baseline top-10 inspected.
✅ Feature importance reviewed.
✅ Ranking overlap calculated.
✅ Error tables saved.

Next: SECTION 5 — Self-check.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.